# Construcción de Score de Propensión a Hey Pro (UC3)

**Objetivo:** Construir un score ponderado de propensión a contratar Hey Pro para priorizar a qué usuarios Havi debe dirigir el mensaje de upselling.

Este score rankea a los usuarios no-Pro considerando factores transaccionales (cashback perdido), demográficos/financieros (ingreso, buró, nómina), actitudinales (NPS, conversaciones) y de engagement (días sin login).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT_DIR = Path().resolve().parent.parent
BASE_TXN = ROOT_DIR / "Datathon_Hey_2026_dataset_transacciones 1" / "dataset_transacciones"
BASE_CONV = ROOT_DIR / "Datathon_Hey_dataset_conversaciones 1" / "dataset_conversaciones"
OUTPUT_DIR = ROOT_DIR / "outputs"

print("Cargando clientes...")
df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})

print("Cargando cashback perdido...")
df_cashback = pd.read_csv(OUTPUT_DIR / "uc3_cashback_perdido.csv")

print("Cargando conversaciones...")
import pyarrow
try:
    pyarrow.unregister_extension_type('pandas.period')
except Exception:
    pass
df_convs = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")

Cargando clientes...
Cargando cashback perdido...
Cargando conversaciones...


## 1. Feature Engineering: Extracción de Señal Conversacional
Identificamos a los usuarios que muestran interés explícito buscando palabras clave en mensajes.

In [2]:
pattern = r"hey\s?pro|beneficio|cash\s?back"
df_convs["menciona_beneficios"] = df_convs["input"].fillna("").str.contains(pattern, case=False, regex=True)

# Etiqueta positiva por usuario
usuarios_interesados = df_convs.groupby("user_id")["menciona_beneficios"].max().reset_index()

print(f"Usuarios que han mencionado interés: {usuarios_interesados['menciona_beneficios'].sum()}")

Usuarios que han mencionado interés: 415


## 2. Consolidación de Features y Tratamiento de Nulos
Unimos toda la información para los usuarios `es_hey_pro == False`.

In [3]:
df_nopro = df_clientes[df_clientes["es_hey_pro"] == False].copy()
print(f"Usuarios sin Hey Pro: {len(df_nopro)}")

# Cruce con Cashback
df_nopro = df_nopro.merge(df_cashback[["user_id", "cashback_perdido_mes"]], on="user_id", how="left")
df_nopro["cashback_perdido_mes"] = df_nopro["cashback_perdido_mes"].fillna(0)

# Cruce con Conversaciones
df_nopro = df_nopro.merge(usuarios_interesados, on="user_id", how="left")
df_nopro["menciona_beneficios"] = df_nopro["menciona_beneficios"].fillna(False).astype(int)

# Tratamiento de nulos (satisfaccion_1_10 según DATA_CONTEXT.md imputar con 7.0)
df_nopro["satisfaccion_1_10"] = df_nopro["satisfaccion_1_10"].fillna(7.0)
df_nopro["ingreso_mensual_mxn"] = df_nopro["ingreso_mensual_mxn"].fillna(df_nopro["ingreso_mensual_mxn"].median())
df_nopro["score_buro"] = df_nopro["score_buro"].fillna(df_nopro["score_buro"].median())
df_nopro["dias_desde_ultimo_login"] = df_nopro["dias_desde_ultimo_login"].fillna(30)

Usuarios sin Hey Pro: 7680


## 3. Cálculo del Score de Propensión
Asignamos pesos de negocio a cada variable. Se utilizan percentiles (`rank(pct=True)`) para estandarizar las magnitudes.

In [4]:
df_nopro["score_propension"] = (
    df_nopro["cashback_perdido_mes"].rank(pct=True) * 35 +
    df_nopro["ingreso_mensual_mxn"].rank(pct=True) * 10 +
    df_nopro["score_buro"].rank(pct=True) * 10 +
    df_nopro["satisfaccion_1_10"].rank(pct=True) * 10 +
    (1 - df_nopro["dias_desde_ultimo_login"].rank(pct=True)) * 10 +
    (df_nopro["nomina_domiciliada"].astype(int)) * 15 +
    (df_nopro["menciona_beneficios"].astype(int)) * 10
)

# Aseguramos escala 0-100
df_nopro["score_propension"] = df_nopro["score_propension"].clip(0, 100).round(2)

## 4. Segmentación y Criterios de Aceptación
Creación de segmentos: **A (>70), B (40-70), C (<40)**

In [5]:
df_nopro["segmento_upsell"] = pd.cut(
    df_nopro["score_propension"], 
    bins=[-1, 40, 70, 101], 
    labels=["C", "B", "A"],
    right=False
)

print("\n--- DISTRIBUCIÓN DEL SCORE POR SEGMENTO ---")
distribucion = df_nopro["segmento_upsell"].value_counts(normalize=True).sort_index(ascending=False) * 100
for idx, pct in distribucion.items():
    count = sum(df_nopro['segmento_upsell'] == idx)
    print(f"Segmento {idx}: {pct:.1f}% ({count} usuarios)")

print("\n--- ACTIVACIÓN EN 1 CLIC (NÓMINA) EN SEGMENTO A ---")
seg_a = df_nopro[df_nopro["segmento_upsell"] == "A"]
pct_nomina_a = seg_a["nomina_domiciliada"].mean() * 100
print(f"% de usuarios en Segmento A con nomina_domiciliada=True: {pct_nomina_a:.1f}%")

print("\n--- VALIDACIÓN: MAYOR SCORE -> MAYOR CASHBACK PERDIDO ---")
val_df = df_nopro.groupby("segmento_upsell", observed=False)["cashback_perdido_mes"].mean().sort_index(ascending=False)
for idx, val in val_df.items():
    print(f"Segmento {idx} (Score Promedio {df_nopro[df_nopro['segmento_upsell']==idx]['score_propension'].mean():.1f}): ${val:.2f} MXN perdidos promedio")


--- DISTRIBUCIÓN DEL SCORE POR SEGMENTO ---
Segmento A: 5.3% (405 usuarios)
Segmento B: 49.8% (3825 usuarios)
Segmento C: 44.9% (3450 usuarios)

--- ACTIVACIÓN EN 1 CLIC (NÓMINA) EN SEGMENTO A ---
% de usuarios en Segmento A con nomina_domiciliada=True: 92.1%

--- VALIDACIÓN: MAYOR SCORE -> MAYOR CASHBACK PERDIDO ---
Segmento A (Score Promedio 74.0): $55.45 MXN perdidos promedio
Segmento B (Score Promedio 55.7): $42.55 MXN perdidos promedio
Segmento C (Score Promedio 26.0): $3.00 MXN perdidos promedio


## 5. Exportación de Resultados
Se guarda la lista rankeada para alimentar las campañas proactivas de Havi.

In [6]:
cols_export = [
    "user_id", "score_propension", "segmento_upsell", "cashback_perdido_mes", 
    "nomina_domiciliada", "menciona_beneficios", "satisfaccion_1_10"
]
df_export = df_nopro.sort_values("score_propension", ascending=False)[cols_export]

export_path = OUTPUT_DIR / "uc3_score_propension.csv"
df_export.to_csv(export_path, index=False)
print(f"Score exportado a: {export_path}")
display(df_export.head(10))

Score exportado a: C:\Users\nanoj\Documents\GitHub\Datathon-2026\outputs\uc3_score_propension.csv


,user_id,score_propension,segmento_upsell,cashback_perdido_mes,nomina_domiciliada,menciona_beneficios,satisfaccion_1_10
2515,USR-04957,86.33,A,33.60,True,1,10.0
3607,USR-07102,83.92,A,44.82,True,1,9.0
3916,USR-07678,82.63,A,71.57,True,0,10.0
5604,USR-10930,82.46,A,36.80,True,0,10.0
1770,USR-03531,82.29,A,48.75,True,0,10.0
3861,USR-07552,82.28,A,85.96,True,0,9.0
1510,USR-03031,82.21,A,55.59,True,1,8.0
1561,USR-03135,82.09,A,84.96,True,0,9.0
6630,USR-12926,81.95,A,137.91,True,0,10.0
902,USR-01776,81.77,A,16.78,True,1,9.0
